# Quiz Maker

Making a quizmaker using the scraper. This is a test environment where i make and see if it works. Then I'll make it a proper python file.

In [1]:
# imports
import os
from dotenv import load_dotenv
from web_scraper import smart_fetch
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [28]:
# system prompt
quiz_system_prompt = """ 
    You are a quiz maker. You are given the url and contents of a site and a topic. The topic is an optional filed. 
    If the topic is given the quiz has to be based on the topic other wise it should be on the entire contents of the page.
    If the title is not relevent to the contents of the page, make the quiz on the entire content of the page.
    The quiz should be 10 questions long with 4 options with 1 correct answer. The difficulty should be divided as 3 easy, 5 medium and 2 hard.
    You will return the following- the quiz title, the source url, the each question with its option and correct ans in the following format:
    {
        "quiz_title": "Example Quiz Title",
        "url": "https://example.com/source",
        "questions": [
            {
                "id": 1,
                "question": "Which planet is known as the Red Planet?",
                "options": [
                    { "id": 1, "content": "Venus" },
                    { "id": 2, "content": "Mars" },
                    { "id": 3, "content": "Jupiter" },
                    { "id": 4, "content": "Mercury" }
                ],
                "answer": 2
                },
                {
                "id": 2,
                "question": "What is the largest ocean on Earth?",
                "options": [
                    { "id": 1, "content": "Atlantic Ocean" },
                    { "id": 2, "content": "Indian Ocean" },
                    { "id": 3, "content": "Arctic Ocean" },
                    { "id": 4, "content": "Pacific Ocean" }
                ],
                "answer": 4
            }
        ]
    }
"""

In [12]:
# user prompt
async def build_user_prompt(url, topic):
    website_contents = await smart_fetch(url)
    user_prompt = f"""
        For the website {url}, having contents {website_contents},
        Please make a quiz on the topic {topic}. 
        If the topic is not given, make a quiz on the entire contents of the page, not incuding irrelevent things such as Terms of Service, Privacy, email links, etc.
        The quiz should returned in JSON format.
    """
    return user_prompt

In [13]:
prompt = await build_user_prompt(
    "https://www.geeksforgeeks.org/python/python-oops-concepts/", 
    "OOPs in Python"
)

print(prompt)


        For the website https://www.geeksforgeeks.org/python/python-oops-concepts/, having contents Python OOP Concepts - GeeksforGeeksSkip to contentCoursesDSA / PlacementsGATE PrepML & Data ScienceDevelopmentCloud / DevOpsProgramming LanguagesAll CoursesTutorialsPythonJavaDSAML & Data ScienceInterview CornerProgramming LanguagesWeb DevelopmentGATECS SubjectsDevOpsSchool LearningSoftware and ToolsPracticePractice Coding ProblemsNation Skillup- Ending Soon!Problem of the DayStudy Abroad ChampionshipJobsApply Now!Post JobsJobs UpdatesApply for Campus MantriNotificationsMark all as readAllView AllNotificationsMark all as readAllUnreadReadYou're all caught up!!Python TutorialData TypesInterview QuestionsExamplesQuizzesDSA PythonData ScienceNumPyPandasPracticeDjangoFlaskProjectsSign In▲Open In AppTechnical ScripterExplorePython FundamentalsPython IntroductionInput and Output in PythonPython VariablesPython OperatorsPython KeywordsPython Data TypesConditional Statements in PythonLoops in P

In [14]:
openai = OpenAI()

In [24]:
async def generate_quiz(url, topic):
    user_prompt = await build_user_prompt(url, topic)
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": quiz_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content

In [31]:
quiz = await generate_quiz(
    "https://www.geeksforgeeks.org/python/python-oops-concepts/",
    "Encapsulation"
)
print(quiz)

{
    "quiz_title": "Python OOP Concepts: Encapsulation",
    "url": "https://www.geeksforgeeks.org/python/python-oops-concepts/",
    "questions": [
        {
            "id": 1,
            "question": "What is encapsulation in OOP?",
            "options": [
                { "id": 1, "content": "Hiding data within an object" },
                { "id": 2, "content": "Using multiple inheritance" },
                { "id": 3, "content": "Creating classes" },
                { "id": 4, "content": "Implementing interfaces" }
            ],
            "answer": 1
        },
        {
            "id": 2,
            "question": "Which of the following is a way to implement encapsulation in Python?",
            "options": [
                { "id": 1, "content": "Using private variables" },
                { "id": 2, "content": "Using multiple classes" },
                { "id": 3, "content": "Using lambda functions" },
                { "id": 4, "content": "Using loops" }
            ]